In [38]:
import pandas as pd

original = "Project2026/data/original/diabetic_data.csv"
original_df = pd.read_csv(original)
ids = original_df["encounter_id"].copy()

train = "train.csv"
test = "test.csv"
train_df = pd.read_csv(train)
test_df = pd.read_csv(test)
ids_test = test_df["id"].copy()

In [ ]:
import pandas as pd
import numpy as np
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import cross_val_predict
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import seaborn as sns
from catboost import CatBoostClassifier

# ── 1. Ladda data ─────────────────────────────────────────────────────────────
original_df = pd.read_csv("Project2026/data/original/diabetic_data.csv")

# ── 2. ICD-9 mapper ───────────────────────────────────────────────────────────
def map_icd9(code):
    try:
        code = str(code)
        if code.startswith('E') or code.startswith('V'):
            return 'external'
        code = int(code.split('.')[0])
        if   1   <= code <= 139: return 'infectious'
        elif 140 <= code <= 239: return 'neoplasms'
        elif 240 <= code <= 279: return 'endocrine'
        elif 280 <= code <= 289: return 'blood'
        elif 290 <= code <= 319: return 'mental'
        elif 320 <= code <= 389: return 'nervous'
        elif 390 <= code <= 459: return 'circulatory'
        elif 460 <= code <= 519: return 'respiratory'
        elif 520 <= code <= 579: return 'digestive'
        elif 580 <= code <= 629: return 'genitourinary'
        elif 630 <= code <= 679: return 'pregnancy'
        elif 680 <= code <= 709: return 'skin'
        elif 710 <= code <= 739: return 'musculoskeletal'
        elif 740 <= code <= 759: return 'congenital'
        elif 760 <= code <= 779: return 'perinatal'
        elif 780 <= code <= 799: return 'symptoms'
        elif 800 <= code <= 999: return 'injury'
        else: return 'other'
    except (ValueError, TypeError):
        return 'other'

# ── 3. Raw clean (ingen imputation/encoding) ──────────────────────────────────
def raw_clean(df):
    df = df.copy()
    df.replace('?', np.nan, inplace=True)

    drop_cols = ["encounter_id", "patient_nbr", "weight",
                 "payer_code", "medical_specialty"]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')

    for col in ['diag_1', 'diag_2', 'diag_3']:
        df[col] = df[col].astype(str).str.split('.').str[0].apply(map_icd9)
          
    df["total_visits"] = (
        df["number_outpatient"] +
        df["number_emergency"] +
        df["number_inpatient"]
    )

    df["high_med"] = (df["num_medications"] > 20).astype(int)

    return df


# def raw_clean(df, is_train=True):
#     df = df.copy()
#     df.replace('?', np.nan, inplace=True)

#     if is_train:
#         # ta bort hospice/dead
#         df = df[~df["discharge_disposition_id"].isin([11, 13, 14])]

#         # deduplicera
#         df = df.sort_values("encounter_id", ascending=False)
#         df = df.drop_duplicates(subset="patient_nbr", keep="first")

#     # droppa kolumner
#     drop_cols = ["encounter_id", "patient_nbr", "weight",
#                  "payer_code", "medical_specialty"]
#     df.drop(columns=drop_cols, inplace=True, errors='ignore')

#     # ICD mapping
#     for col in ['diag_1', 'diag_2', 'diag_3']:
#         df[col] = df[col].astype(str).str.split('.').str[0].apply(map_icd9)
        
#     df["total_visits"] = (
#         df["number_outpatient"] +
#         df["number_emergency"] +
#         df["number_inpatient"]
#     )
    
#     df["high_med"] = (df["num_medications"] > 20).astype(int)

#    return df


# ── 4. Förbered data ──────────────────────────────────────────────────────────
cleaned_train = raw_clean(train_df)
cleaned_test = raw_clean(test_df)

X_train = cleaned_train.drop("readmitted", axis=1)
y_train = cleaned_train["readmitted"]

X_test = cleaned_test

# cleaned_org = raw_clean(original_df)
# X_org = cleaned_org.drop("readmitted", axis=1)
# y_org = cleaned_org["readmitted"]

# ── Correlation analysis ─────────────────────────────────────────


# Kopia av data
# df_corr = cleaned_train.copy()

# # Gör target numerisk
# target_map = {'NO': 0, '>30': 1, '<30': 2}
# df_corr['readmitted_num'] = df_corr['readmitted'].map(target_map)

# # Bara numeriska kolumner
# numeric_df = df_corr.select_dtypes(include=np.number)

# # Correlation med target
# corr = numeric_df.corr()["readmitted_num"].sort_values()

# plt.figure(figsize=(6,10))
# corr.drop("readmitted_num").plot(kind='barh')
# plt.title("Correlation with Readmitted")
# plt.xlabel("Correlation")
# plt.tight_layout()
# plt.show()

# plt.figure(figsize=(10,8))
# sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
# plt.title("Correlation Heatmap (Numerical Features)")
# plt.show()

# 80/20 split, stratifierad så klassbalansen bevaras i båda delar
# X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
#     X, y, ids,
#     test_size=0.2,
#     stratify=y,
#     random_state=42
# )

#print(f"Träning: {X_train.shape[0]} rader | Test: {X_test.shape[0]} rader")
print(f"Klassdistribution träning:\n{y_train.value_counts(normalize=True).round(3)}")

# ── 5. Kolumntyper ────────────────────────────────────────────────────────────
AGE_CATS = ['[0-10)','[10-20)','[20-30)','[30-40)','[40-50)',
            '[50-60)','[60-70)','[70-80)','[80-90)','[90-100)']
GLU_CATS = ['Norm', '>200', '>300']
A1C_CATS = ['Norm', '>7', '>8']

ORDINAL_COLS = ['age', 'max_glu_serum', 'A1Cresult']
ORDINAL_CATS = [AGE_CATS, GLU_CATS, A1C_CATS]

NOMINAL_COLS = [
    'race', 'gender', 'admission_type_id',
    'discharge_disposition_id', 'admission_source_id',
    'change', 'diabetesMed',
    'diag_1', 'diag_2', 'diag_3',
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

NUMERIC_COLS = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

# ── 6. Preprocessor ───────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler',  StandardScaler()),
        ]), NUMERIC_COLS),

        ('ord', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(
                categories=ORDINAL_CATS,
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )),
        ]), ORDINAL_COLS),

        ('nom', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(
                drop='first',
                sparse_output=False,
                handle_unknown='ignore'
            )),
        ]), NOMINAL_COLS),
    ],
    remainder='drop'
)

# ── 7. Modeller ───────────────────────────────────────────────────────────────
models = {
    # "Linear SVC": Pipeline([
    #     ('pre', preprocessor),
    #     ('smote', SMOTE(sampling_strategy='not majority')),
    #     ('clf', LinearSVC(
    #         dual=False,
    #         #class_weight='balanced',
    #         max_iter=2000, random_state=42
    #     )),
    # ]),
    # "Random Forest": Pipeline([
    #     ('pre', preprocessor),
    #     #('smote', SMOTE(sampling_strategy='not majority')),
    #     ('clf', RandomForestClassifier(
    #         class_weight={'<30':8, '>30':2, 'No':1},
    #         n_estimators=500,
    #         n_jobs=-1, random_state=42
    #     )),
    # ]),
    "CatBoostClassifier": Pipeline([
        ('pre', preprocessor),
        ('clf', CatBoostClassifier(
            iterations=100,      
            learning_rate=0.1,   
            depth=6,              
            verbose=0 
        )),
    ]),
}

# ── 8. CV + slutlig utvärdering ───────────────────────────────────────────────
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# SCORE_METRICS = ['accuracy', 'balanced_accuracy', 'f1_macro', 'precision_macro', 'recall_macro']

for name, pipe in models.items():
#     print(f"\n{'='*50}")
#     print(f"  {name}")
#     print('='*50)

#     cv_results = cross_validate(
#         pipe, X_org, y_org,
#         cv=cv, scoring=SCORE_METRICS,
#         n_jobs=-1
#     )
#     for metric in SCORE_METRICS:
#         scores = cv_results[f'test_{metric}']
#         print(f"  CV {metric:<25} {scores.mean():.4f} ± {scores.std():.4f}")

#     #Slutlig test på hållen testdata
#     y_pred = cross_val_predict(pipe, X_org, y_org, cv=cv, n_jobs=-1)
#     print(classification_report(y_org, y_pred))
    # fig, ax = plt.subplots(figsize=(6, 6))

    # ConfusionMatrixDisplay.from_predictions(
    #     y_org,
    #     y_pred,
    #     display_labels=np.unique(y_org),
    #     cmap='Blues',
    #     values_format=".2f",
    #     normalize='true',   # <-- viktigt för rapport!
    #     ax=ax
    # )

    # plt.title(f"Confusion Matrix ({name})")
    # plt.tight_layout()
    # plt.show()
    
    #print(confusion_matrix(y_org, y_pred))

    # sample_weights = compute_sample_weight(
    #     class_weight='balanced',
    #     y=y_train
    # )

    pipe.fit(X_train, y_train)
        
    y_pred = pipe.predict(X_test).flatten()
    
    results = pd.DataFrame({
        "id": ids_test,
        "readmitted": y_pred
    })
    results.to_csv("predictions.csv", index=False)
    # print(f"\n  Testset resultat:")
    # print(classification_report(y_test, y_pred, zero_division=0))
    # print("  Confusion matrix:")
    # print(confusion_matrix(y_test, y_pred))

Klassdistribution träning:
readmitted
No     0.539
>30    0.349
<30    0.112
Name: proportion, dtype: float64


In [51]:
import pandas as pd
import numpy as np
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import cross_val_predict
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import seaborn as sns
from catboost import CatBoostClassifier

# ── 1. Ladda data ─────────────────────────────────────────────────────────────
original_df = pd.read_csv("Project2026/data/original/diabetic_data.csv")

# ── 2. ICD-9 mapper ───────────────────────────────────────────────────────────
def map_icd9(code):
    try:
        code = str(code)
        if code.startswith('E') or code.startswith('V'):
            return 'external'
        code = int(code.split('.')[0])
        if   1   <= code <= 139: return 'infectious'
        elif 140 <= code <= 239: return 'neoplasms'
        elif 240 <= code <= 279: return 'endocrine'
        elif 280 <= code <= 289: return 'blood'
        elif 290 <= code <= 319: return 'mental'
        elif 320 <= code <= 389: return 'nervous'
        elif 390 <= code <= 459: return 'circulatory'
        elif 460 <= code <= 519: return 'respiratory'
        elif 520 <= code <= 579: return 'digestive'
        elif 580 <= code <= 629: return 'genitourinary'
        elif 630 <= code <= 679: return 'pregnancy'
        elif 680 <= code <= 709: return 'skin'
        elif 710 <= code <= 739: return 'musculoskeletal'
        elif 740 <= code <= 759: return 'congenital'
        elif 760 <= code <= 779: return 'perinatal'
        elif 780 <= code <= 799: return 'symptoms'
        elif 800 <= code <= 999: return 'injury'
        else: return 'other'
    except (ValueError, TypeError):
        return 'other'

# ── 3. Raw clean (ingen imputation/encoding) ──────────────────────────────────
def raw_clean(df):
    df = df.copy()
    df.replace('?', np.nan, inplace=True)

    drop_cols = ["encounter_id", "patient_nbr", "weight",
                 "payer_code", "medical_specialty"]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')

    for col in ['diag_1', 'diag_2', 'diag_3']:
        df[col] = df[col].astype(str).str.split('.').str[0].apply(map_icd9)
          
    df["total_visits"] = (
        df["number_outpatient"] +
        df["number_emergency"] +
        df["number_inpatient"]
    )

    df["high_med"] = (df["num_medications"] > 20).astype(int)

    return df


# ── 4. Förbered data ──────────────────────────────────────────────────────────

cleaned_org = raw_clean(original_df)
X_org = cleaned_org.drop("readmitted", axis=1)
y_org = cleaned_org["readmitted"]

#print(f"Träning: {X_train.shape[0]} rader | Test: {X_test.shape[0]} rader")
print(f"Klassdistribution träning:\n{y_org.value_counts(normalize=True).round(3)}")

# ── 5. Kolumntyper ────────────────────────────────────────────────────────────
AGE_CATS = ['[0-10)','[10-20)','[20-30)','[30-40)','[40-50)',
            '[50-60)','[60-70)','[70-80)','[80-90)','[90-100)']
GLU_CATS = ['None','Norm', '>200', '>300']
A1C_CATS = ['None', 'Norm', '>7', '>8']

ORDINAL_COLS = ['age', 'max_glu_serum', 'A1Cresult']
ORDINAL_CATS = [AGE_CATS, GLU_CATS, A1C_CATS]

NOMINAL_COLS = [
    'race', 'gender', 'admission_type_id',
    'discharge_disposition_id', 'admission_source_id',
    'change', 'diabetesMed',
    'diag_1', 'diag_2', 'diag_3',
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

NUMERIC_COLS = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

# ── 6. Preprocessor ───────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler',  StandardScaler()),
        ]), NUMERIC_COLS),

        ('ord', Pipeline([
            #('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(
                categories=ORDINAL_CATS,
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )),
        ]), ORDINAL_COLS),

        ('nom', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(
                drop='first',
                sparse_output=False,
                handle_unknown='ignore'
            )),
        ]), NOMINAL_COLS),
    ],
    remainder='drop'
)

# ── SMOTE visualisering ─────────────────────────────────────────────

# fig, axes = plt.subplots(2, 1, figsize=(8, 12))  # <-- större figur

# colors = {
#     'NO': 'steelblue',
#     '>30': 'orange',
#     '<30': 'green'
# }

# # ── Före SMOTE ─────────────────────────────
# counts = y_org.value_counts()
# counts.plot(
#     kind='bar',
#     ax=axes[0],
#     color=[colors[c] for c in counts.index]
# )

# axes[0].set_title("Before SMOTE", fontsize=14)
# axes[0].set_xlabel("Class", fontsize=12)
# axes[0].set_ylabel("Count", fontsize=12)
# axes[0].tick_params(axis='x', rotation=0)

# # ── Efter SMOTE ────────────────────────────
# counts_res = pd.Series(y_res).value_counts()
# counts_res.plot(
#     kind='bar',
#     ax=axes[1],
#     color=[colors[c] for c in counts_res.index]
# )

# axes[1].set_title("After SMOTE", fontsize=14)
# axes[1].set_xlabel("Class", fontsize=12)
# axes[1].set_ylabel("Count", fontsize=12)
# axes[1].tick_params(axis='x', rotation=0)

# # ── Layout fix ─────────────────────────────
# plt.tight_layout()

# # (valfri men BRA för rapport)
# plt.savefig("class_distribution_smote.png", dpi=300)

# plt.show()
# ── 7. Modeller ───────────────────────────────────────────────────────────────
models = {
    "Linear SVC": Pipeline([
        ('pre', preprocessor),
        ('smote', SMOTE(sampling_strategy='not majority')),
        ('clf', LinearSVC(
            dual=False,
            #class_weight='balanced',
            max_iter=2000, random_state=42
        )),
    ]),
    # "Random Forest": Pipeline([
    #     ('pre', preprocessor),
    #     ('smote', SMOTE(sampling_strategy='not majority')),
    #     ('clf', RandomForestClassifier(
    #         n_estimators=500,
    #         n_jobs=-1, random_state=42
    #     )),
    # ]),
    # "CatBoostClassifier": Pipeline([
    #     ('pre', preprocessor),
    #     ('smote', SMOTE(sampling_strategy='not majority')),
    #     ('clf', CatBoostClassifier(
    #         iterations=100,      
    #         learning_rate=0.1,   
    #         depth=6,              
    #         verbose=0 
    #     )),
    # ]),
}

# ── 8. CV + slutlig utvärdering ───────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORE_METRICS = ['accuracy', 'balanced_accuracy', 'f1_macro', 'precision_macro', 'recall_macro']

for name, pipe in models.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)

    cv_results = cross_validate(
        pipe, X_org, y_org,
        cv=cv, scoring=SCORE_METRICS,
        n_jobs=-1
    )
    for metric in SCORE_METRICS:
        scores = cv_results[f'test_{metric}']
        print(f"  CV {metric:<25} {scores.mean():.4f} ± {scores.std():.4f}")

    #Slutlig test på hållen testdata
    y_pred = cross_val_predict(pipe, X_org, y_org, cv=cv, n_jobs=-1)
    print(classification_report(y_org, y_pred))

    
#     results = pd.DataFrame({
#         "id": ids_test,
#         "readmitted": y_pred
#     })
#     results.to_csv("predictions.csv", index=False)


Klassdistribution träning:
readmitted
NO     0.539
>30    0.349
<30    0.112
Name: proportion, dtype: float64

  Linear SVC


Python(5659) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(5660) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(5661) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(5662) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(5663) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(5664) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(5665) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(5666) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  CV accuracy                  0.4832 ± 0.0030
  CV balanced_accuracy         0.4551 ± 0.0047
  CV f1_macro                  0.4267 ± 0.0036
  CV precision_macro           0.4345 ± 0.0030
  CV recall_macro              0.4551 ± 0.0047
              precision    recall  f1-score   support

         <30       0.19      0.43      0.26     11357
         >30       0.44      0.37      0.40     35545
          NO       0.67      0.57      0.62     54864

    accuracy                           0.48    101766
   macro avg       0.43      0.45      0.43    101766
weighted avg       0.54      0.48      0.50    101766

